# T02 — Ingestion und Chunking von Dokumenten

Wie aus Rohdokumenten die Chunks werden, die die Pipeline aus T01 findet:

1. PDF-Parsing: MarkItDown vs. docling
2. Chunking-Strategien: nach Überschriften, feste Grösse, absatzbasiert
3. Chunk-Grössen und das Token-Limit des Embedding-Modells
4. Tabellen und Tabellenblätter
5. Wo ein Fakt bei jeder Strategie landet

Kein API-Key nötig. Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Setup

In [ ]:
import re
import time

from IPython.display import display

from conversational_toolkit.chunking.excel_chunker import ExcelChunker
from conversational_toolkit.chunking.pdf_chunker import (
    MarkdownConverterEngine,
    PDFChunker,
)
from ki_kompass_rag.baseline_rag import DATA_DIR
from ki_kompass_rag.ingestion import (
    analyze_chunks,
    char_histogram,
    estimate_tokens,
    fixed_size_chunks,
    header_based_chunks,
    paragraph_aware_chunks,
)

SAMPLE_PDF = str(DATA_DIR / "EPD_pallet_relicyc_logypal1.pdf")

## 1. PDF-Parsing: MarkItDown vs. Docling

Ein PDF muss vor dem Chunking in Text (hier Markdown) umgewandelt werden. Der Parser entscheidet, welche Überschriften erkannt werden, ob Tabellen erhalten bleiben und welcher Text verloren geht.

| Engine | Ansatz | Stärken | Schwächen |
|---|---|---|---|
| `markitdown` | Liest die Textebene aus | Schnell, viele Formate (DOCX, PPTX) | Verliert oft Überschriften und Tabellenstruktur |
| `docling` | Layout- und Tabellenmodelle | Überschriften, Tabellen, Lesereihenfolge | Langsamer; lädt beim ersten Aufruf Modelle herunter |

Die Pipeline verwendet docling ohne OCR (alle PDFs hier haben eine Textebene). `load_chunks()` überspringt ausserdem Dateien über 20 MB, die den Arbeitsspeicher sprengen können.

In [ ]:
chunker = PDFChunker(do_ocr=False)
markdown = {}
for engine in MarkdownConverterEngine:
    start = time.perf_counter()
    markdown[engine.value] = chunker.to_markdown(SAMPLE_PDF, engine=engine)
    seconds = time.perf_counter() - start
    print(
        f"{engine.value:<12}{len(markdown[engine.value]):>8,} chars  {seconds:5.1f} s"
    )

Die Zeit für docling enthält das einmalige Laden der Modelle. Überschriften und Tabellen vergleichen:

In [ ]:
def headings(md: str) -> list[str]:
    return [line.strip() for line in md.splitlines() if re.match(r"^#{1,4} ", line)]


def count_tables(md: str) -> int:
    lines = [line.lstrip() for line in md.splitlines()]
    return sum(
        line.startswith("|") and not prev.startswith("|")
        for prev, line in zip(["", *lines], lines)
    )


for name, md in markdown.items():
    print(f"{name}: {len(headings(md))} headings, {count_tables(md)} tables")
    for heading in headings(md)[:10]:
        print(f"  {heading}")

In [ ]:
table = []
for line in markdown["docling"].splitlines():
    if line.lstrip().startswith("|"):
        table.append(line)
    elif table:
        break
print("\n".join(table) or "No table found.")

## 2. Chunking-Strategien

| Strategie | Wie sie teilt | Geeignet für | Risiko |
|---|---|---|---|
| Nach Überschriften | Ein Chunk pro Markdown-Überschrift | Strukturierte Dokumente (EPDs, Richtlinien) | Sehr grosse oder winzige Chunks bei wenigen Überschriften |
| Feste Grösse | Alle N Zeichen, mit Überlappung | Vorhersehbare Grössen | Schneidet mitten im Satz oder in der Tabelle |
| Absatzbasiert | Fasst Absätze bis zu einer Zielgrösse zusammen | Fliesstext | Grössen schwanken stärker als bei fester Grösse |

In [ ]:
chunkings = {
    "header_based": header_based_chunks(SAMPLE_PDF),
    "fixed_size_800": fixed_size_chunks(SAMPLE_PDF, chunk_size=800, overlap=100),
    "paragraph_500": paragraph_aware_chunks(SAMPLE_PDF, target_chars=500),
}
for name, chunks in chunkings.items():
    print(analyze_chunks(chunks, name))

In [ ]:
for name, chunks in chunkings.items():
    display(char_histogram(chunks, title=name))

## 3. Das Token-Limit des Embedding-Modells

Ein Embedding-Modell liest nur eine begrenzte Anzahl Tokens; der Rest eines längeren Chunks wird ignoriert. `text-embedding-3-small` liest 8191 Tokens, die Anzahl `>8191 tokens` oben ist also normalerweise 0. Kleine lokale Modelle sind viel strenger: `all-MiniLM-L6-v2` hört nach 256 Tokens (~1000 Zeichen) auf:

In [ ]:
for name, chunks in chunkings.items():
    over = sum(estimate_tokens(c.content) > 256 for c in chunks)
    print(f"{name:<15}{over:>4} / {len(chunks)} chunks over 256 tokens")

Mit einem solchen Modell eine kleinere Chunk-Grösse oder Chunking mit fester Grösse verwenden, um die Länge zu begrenzen.

## 4. Tabellen und Tabellenblätter

Docling gibt PDF-Tabellen als Markdown-Tabellen aus, die in Chunks nach Überschriften intakt bleiben:

In [ ]:
table_chunks = [c for c in chunkings["header_based"] if c.content.count("|") >= 4]
print(f"{len(table_chunks)} header-based chunks contain a table\n")
if table_chunks:
    print(table_chunks[0].content)

Excel-Dateien werden separat gechunkt, eine Markdown-Tabelle pro Tabellenblatt:

In [ ]:
for c in ExcelChunker().make_chunks(str(DATA_DIR / "ART_product_overview.xlsx")):
    print(f"Sheet: {c.title}")
    print(c.content)

## 5. Wo die Fakten landen

Ein Retriever kann nur liefern, was ein Chunk enthält. Welche Chunks erwähnen das GWP, und wie viel Kontext tragen sie mit?

In [ ]:
terms = ["gwp", "global warming", "carbon"]
for name, chunks in chunkings.items():
    hits = [c for c in chunks if any(t in c.content.lower() for t in terms)]
    print(f"\n{name}: {len(hits)} / {len(chunks)} chunks mention {terms}")
    if hits:
        print(f"First hit ({len(hits[0].content)} chars): {hits[0].content[:500]!r}")

Chunks nach Überschriften halten einen Abschnitt zusammen; Chunks fester Grösse trennen unter Umständen eine Tabelle von ihrer Überschrift.

## Zusammenfassung

| | Nach Überschriften | Feste Grösse | Absatzbasiert |
|---|---|---|---|
| Grenzen | Überschriften | Alle N Zeichen | Absatzumbrüche |
| Vorhersehbare Grösse | Gering | Hoch | Mittel |
| Kontext pro Chunk | Hoch | Mittel (Überlappung hilft) | Mittel |

Docling mit Chunking nach Überschriften passt zu den strukturierten EPDs und Richtlinien in diesem Korpus. Für weniger strukturierte Dokumente oder kleine Embedding-Modelle ist Chunking mit fester Grösse sicherer.